In [1]:
import sys

import torch

import pandas as pd

from config.feature_config import FeatureConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from dice4el.scenario.scenario_handler import ScenarioHandler

from dice4el.scenario.scenario_model import ScenarioLSTM
from dice4el.scenario.scenario_model_wrapper import ScenarioModelWrapper

from dice4el.dice4el_config import EventLogDiCEConfig
from dice4el.eventlog_dice import EventLogDiCE
from dice4el.eventlog_dice_optimized import EventLogDiCEOptimized

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=42)

In [3]:
excel_file = pd.ExcelFile("../../../data/bpic19.xlsx", engine="openpyxl")

df = pd.concat(
    [
        pd.read_excel(
            excel_file,
            keep_default_na=False,
            dtype={
                "case:concept:name": "string",
                "concept:name": "string",
                "case:Spend area text": "string",
                "case:Document Type": "string",
                "case:Sub spend area text": "string",
                "case:Purch. Doc. Category name": "string",
                "case:Item Type": "string",
                "case:Item Category": "string",
                "case:Spend classification text": "string",
                "case:Source": "string",
                "case:GR-Based Inv. Verif.": "string",
                "case:Goods Receipt": "string",
                "Cumulative net worth (EUR)": "float32",
                "time_delta": "float32",
            }
        )
        for sheet in excel_file.sheet_names
    ],
    ignore_index=True,
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,Cumulative net worth (EUR),case:Document Type,case:GR-Based Inv. Verif.,case:Goods Receipt,case:Item Category,case:Item Type,case:Purch. Doc. Category name,case:Source,case:Spend area text,case:Spend classification text,case:Sub spend area text,concept:name,time_delta
0,2000000000_00001,2018-01-02 12:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Created,0.0
1,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Complete,3600.0
2,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Awaiting Approval,0.0
3,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Document Completed,0.0
4,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: In Transfer to Execution Syst.,0.0
5,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Ordered,0.0
6,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Change was Transmitted,0.0
7,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,Create Purchase Order Item,0.0
8,2000000000_00001,2018-01-02 22:59:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,Vendor creates invoice,32760.0
9,2000000000_00001,2018-03-06 06:44:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,Record Goods Receipt,5384700.0


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['Cumulative net worth (EUR)', 'case:Document Type', 'case:GR-Based Inv. Verif.', 'case:Goods Receipt', 'case:Item Category', 'case:Item Type', 'case:Purch. Doc. Category name', 'case:Source', 'case:Spend area text', 'case:Spend classification text', 'case:Sub spend area text', 'concept:name', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
case:Document Type             categorical    case     yes    ['EC Purchase order', 'Framework order', 'Standard PO'] N/A        data_derived        
case:GR-Based Inv. Verif.      categorical    case     yes    ['False', 'True

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

In [11]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [12]:
scenario_model = ScenarioLSTM.load()

In [13]:
scenario_model_wrapper = ScenarioModelWrapper(
    scenario_model=scenario_model,
    scenario_handler=scenario_handler,
    device=device
)

### --- Process Constraints ---

In [14]:
engine = ProcessModelConstraintEngine.load(
     path = "../pretrained_models/"
)

In [15]:
engine.parallel_sets

[{'Change Delivery Indicator', 'Change Quantity'},
 {'Cancel Invoice Receipt',
  'Cancel Subsequent Invoice',
  'Clear Invoice',
  'Record Invoice Receipt',
  'Record Subsequent Invoice',
  'Remove Payment Block'},
 {'Cancel Invoice Receipt', 'Cancel Subsequent Invoice'},
 {'Change Currency', 'Change Price', 'Change payment term'},
 {'SRM: Change was Transmitted', 'SRM: Ordered'}]

In [16]:
engine.branching_sets

[{'Block Purchase Order Item',
  'Cancel Goods Receipt',
  'Cancel Invoice Receipt',
  'Cancel Subsequent Invoice',
  'Change Approval for Purchase Order',
  'Change Currency',
  'Change Delivery Indicator',
  'Change Price',
  'Change Quantity',
  'Change Rejection Indicator',
  'Change Storage Location',
  'Change payment term',
  'Clear Invoice',
  'Create Purchase Order Item',
  'Create Purchase Requisition Item',
  'Delete Purchase Order Item',
  'Reactivate Purchase Order Item',
  'Receive Order Confirmation',
  'Record Goods Receipt',
  'Record Invoice Receipt',
  'Record Service Entry Sheet',
  'Record Subsequent Invoice',
  'Release Purchase Order',
  'Remove Payment Block',
  'SRM: Awaiting Approval',
  'SRM: Change was Transmitted',
  'SRM: Complete',
  'SRM: Created',
  'SRM: Document Completed',
  'SRM: Held',
  'SRM: In Transfer to Execution Syst.',
  'SRM: Incomplete',
  'SRM: Ordered',
  'Vendor creates debit memo',
  'Vendor creates invoice'},
 {'Record Invoice Receipt

### --- Load Experiments ---

In [17]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic19-cf_generated_experiments_dice4el_output.txt", console=False)

In [18]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [19]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

### --- Counterfactuals ---

In [20]:
dice4el_config = EventLogDiCEConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
    w_margin_loss=1.0,
    w_scenario_loss=1.0,
    w_distance_loss=1.0,
    w_cat_loss=1.0,
)
dice4el_config.validate()

In [21]:
cf_DiCE4EL = EventLogDiCE(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results = generator.run_experiment_df(
    cf_method=cf_DiCE4EL,
    technique="DiCE4EL",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/500 [00:00<?, ?case/s]

In [22]:
results

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,4508042805_00010,6,1,2,0.051759,0.003517,0.1,0.214286,0.600000,...,1.664702,0.600000,0.051759,0.1,0.003517,0.214286,0.798657,0.798657,1.000000,1.000000
1,0,4508044035_00320,8,1,2,0.011430,0.022860,0.0,0.214286,0.578947,...,1.413729,0.578947,0.011430,0.0,0.022860,0.214286,0.609066,0.609066,0.999997,0.999997
2,0,4507029240_00300,10,1,4,0.135290,0.070580,0.2,0.357143,0.739130,...,1.858061,0.739130,0.135290,0.2,0.070580,0.357143,0.626497,0.626497,0.999998,0.999998
3,0,4507017655_00360,12,1,6,0.148813,0.097625,0.2,0.357143,0.777778,...,1.937687,0.777778,0.148813,0.2,0.097625,0.357143,0.653954,0.653954,0.999998,0.999998
4,0,4507037320_00020,14,1,6,0.000055,0.000110,0.0,0.214286,0.516129,...,0.730470,0.516129,0.000055,0.0,0.000110,0.214286,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
462,49,4507026467_00001,40,1,0,0.000428,0.000857,0.0,0.562500,0.000000,...,0.562928,0.000000,0.000428,0.0,0.000857,0.562500,0.000000,0.000000,0.000000,0.000000
463,49,4507032796_00001,42,1,0,0.022736,0.045472,0.0,0.625000,0.425287,...,2.005715,0.425287,0.022736,0.0,0.045472,0.625000,0.932692,0.000000,1.000000,0.000000
464,49,4507024372_00001,44,1,0,0.013669,0.027338,0.0,0.593750,0.890110,...,2.435676,0.890110,0.013669,0.0,0.027338,0.593750,0.938147,0.000000,1.000000,0.000000
465,49,4507026083_00001,48,1,0,0.000309,0.000618,0.0,0.562500,0.000000,...,0.562809,0.000000,0.000309,0.0,0.000618,0.562500,0.000000,0.000000,0.000000,0.000000


In [23]:
cf_DiCE4EL_optim = EventLogDiCEOptimized(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results_optim = generator.run_experiment_df(
    cf_method=cf_DiCE4EL_optim,
    technique="DiCE4EL-Optimized",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/500 [00:00<?, ?case/s]

In [24]:
results_optim

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,4508042805_00010,6,1,2,0.278894,0.257788,0.3,0.357143,0.400000,...,1.508776,0.400000,0.278894,0.3,0.257788,0.357143,0.472740,0.472740,0.000000,0.000000
1,0,4508044035_00320,8,1,2,0.147204,0.194408,0.1,0.214286,0.789474,...,2.016393,0.789474,0.147204,0.1,0.194408,0.214286,0.865429,0.000000,1.000000,0.000000
2,0,4507029240_00300,10,1,4,0.178886,0.157771,0.2,0.285714,0.521739,...,0.986339,0.521739,0.178886,0.2,0.157771,0.285714,0.000000,0.704803,0.000000,0.999999
3,0,4507017655_00360,12,1,6,0.289354,0.278709,0.3,0.357143,0.555556,...,1.863244,0.555556,0.289354,0.3,0.278709,0.357143,0.661191,0.661191,0.999998,0.999998
4,0,4507037320_00020,14,1,6,0.003773,0.007546,0.0,0.142857,0.516129,...,0.662759,0.516129,0.003773,0.0,0.007546,0.142857,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
462,49,4507026467_00001,40,1,0,0.002175,0.004350,0.0,0.375000,0.000000,...,0.377175,0.000000,0.002175,0.0,0.004350,0.375000,0.000000,0.000000,0.000000,0.000000
463,49,4507032796_00001,42,1,0,0.025486,0.050972,0.0,0.343750,0.000000,...,0.369236,0.000000,0.025486,0.0,0.050972,0.343750,0.000000,0.000000,0.000000,0.000000
464,49,4507024372_00001,44,1,0,0.014659,0.029317,0.0,0.343750,0.890110,...,2.186550,0.890110,0.014659,0.0,0.029317,0.343750,0.938031,0.000000,1.000000,0.000000
465,49,4507026083_00001,48,1,0,0.003417,0.006834,0.0,0.343750,0.000000,...,0.347167,0.000000,0.003417,0.0,0.006834,0.343750,0.000000,0.000000,0.000000,0.000000


### --- Cleanup ---

In [25]:
sys.stdout = original_stdout
log_file.close()